#####Foreign catalog connection with databricks

In [0]:
%sql
--one tiem operation to create connection
CREATE CONNECTION fc_mysql_conn
TYPE MYSQL
OPTIONS (
  host '34.123.166.158',
  port '3306',
  user 'devuser',
  password 'lets hide this password before commit to git'
)

In [0]:
%sql 
DROP CATALOG fc_mysql_catalog;
CREATE FOREIGN CATALOG fc_mysql_catalog
USING CONNECTION fc_mysql_conn;

In [0]:
%sql
--checking the connection
DESCRIBE CONNECTION fc_mysql_conn;

In [0]:
%sql
show catalogs;

In [0]:
%sql
select * from fc_mysql_catalog.logistics.shipments1;

In [0]:
%sql
--now create the bronze table and fetch the data from FC, unity catalog only stores the metadata  and not actual data.
CREATE TABLE IF NOT EXISTS lakeflow_pl_cat.lakeflow_pl_sch.fc_bronze_shipments
USING DELTA
AS SELECT 
shipment_id,
first_name,
last_name,
age,
role,
updated_at
FROM fc_mysql_catalog.logistics.shipments;

In [0]:
%sql
-- check if data is moved
select * from lakeflow_pl_cat.lakeflow_pl_sch.fc_bronze_shipments;


#####INSERT INTO logistics.shipments1 VALUES (5000006,'Bala','Chander',35,'DE',CURRENT_TIMESTAMP); ####Update data into the source Database table and run the incremental load update logistics.shipments1 set role='Databricks Data Engineer',updated_at=CURRENT_TIMESTAMP where shipment_id=5000006;

In [0]:
%sql
select coalesce(max(updated_at),'1990-01-01')
from lakeflow_pl_cat.lakeflow_pl_sch.fc_bronze_shipments;

In [0]:
%sql
--performing CDC/Incremental load & SCD
INSERT INTO lakeflow_pl_cat.lakeflow_pl_sch.fc_bronze_shipments
SELECT
shipment_id,
first_name,
last_name,
age,
role,
updated_at
FROM fc_mysql_catalog.logistics.shipments
WHERE updated_at > (
  SELECT coalesce(MAX(updated_at),'1990-01-01')
  FROM lakeflow_pl_cat.lakeflow_pl_sch.fc_bronze_shipments
);

In [0]:
%sql
select * from lakeflow_pl_cat.lakeflow_pl_sch.fc_bronze_shipments;

In [0]:
%sql
--latest version or history you can access
SELECT *, row_number() over(partition by shipment_id order by updated_at desc) rno 
FROM lakeflow_pl_cat.lakeflow_pl_sch.fc_bronze_shipments
QUALIFY rno > 1;

In [0]:
%sql
--incremental load Merge for SCD type 1
MERGE INTO lakeflow_pl_cat.lakeflow_pl_sch.fc_bronze_shipments AS target
USING fc_mysql_catalog.logistics.shipments AS source
ON target.shipment_id = source.shipment_id
WHEN MATCHED THEN
UPDATE SET
target.shipment_id = source.shipment_id
target.first_name = source.first_name
target.last_name = source.last_name
target.age = source.age
target.role = source.role
target.updated_at = source.updated_at
WHEN NOT MATCHED THEN
INSERT (shipment_id,first_name,last_name,age,role,updated_at)
VALUES(source.shipment_id,
       source.first_name,
       source.last_name,
       source.age,
       source.role,
       source.updated_at)
WHEN NOT MATCHED BY SOURCE THEN
DELETE;

In [0]:
%sql
delete from lakeflow_pl_cat.lakeflow_pl_sch.fc_bronze_shipments
where shipment_id in (select tgt.shipment_id from lakeflow_pl_cat.lakeflow_pl_sch.fc_bronze_shipments tgt left join fc_mysql_catalog.logistics.shipments src on tgt.shipment_id=src.shipment_id where src.shipment_id is null);

In [0]:
sql
select * from lakeflow_pl_cat.lakeflow_pl_sch.fc_bronze_shipments;